In [ ]:
%load_ext autoreload
%autoreload 2

# Flight Delay ML Platform — Historical Model Ablation Study

## Research Objective

Investigate whether point-in-time historical delay statistics provide incremental predictive value beyond scheduled flight information.

## Research Questions

- Do origin-level historical delay features improve prediction performance?
- Do airline-origin historical features provide additional predictive value?
- Are historical features equally beneficial for departure and arrival delay prediction?
- Do historical features improve temporal generalization?

## Experimental Design

Three feature configurations are compared:

- V1: Scheduled baseline features (17 features)
- V2-A: Baseline + origin historical features (19 features)
- V2-B: Baseline + all historical features (21 features)

Each configuration is evaluated using Logistic Regression and LightGBM.

## Evaluation Protocol

- Training: January–December 2023
- Validation: January–August 2024
- Primary metric: Average Precision (AP)
- Secondary metrics: ROC-AUC, F1, Precision, Recall, Brier Score

Model hyperparameters remain unchanged during the initial ablation study.

Calibration, threshold-selection, and final test datasets remain reserved for subsequent experiments.

Historical feature availability is based on the 120-day research-proxy policy rather than verified historical publication timestamps.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import lightgbm

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

assert (ROOT / "pyproject.toml").is_file()

FEATURE_DIR = (
    ROOT / "data/interim/historical_features_v1"
)

PROCESSED_DIR = ROOT / "data/processed"

print("=== Environment ===")
print("Project root:", ROOT)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("LightGBM:", lightgbm.__version__)

assert FEATURE_DIR.is_dir()
assert PROCESSED_DIR.is_dir()

In [ ]:
from flight_delay.data.manifest import sha256_file

from flight_delay.features.historical_features import (
    HISTORICAL_FEATURE_COLUMNS,
)

with (
    FEATURE_DIR / "feature_manifest.json"
).open(encoding="utf-8") as f:
    feature_manifest = json.load(f)

identity_columns = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
]

history_columns = list(HISTORICAL_FEATURE_COLUMNS)

assert feature_manifest["feature_version"] == (
    "historical_features_v1"
)

assert feature_manifest["identity_columns"] == identity_columns
assert feature_manifest["feature_columns"] == history_columns

config = feature_manifest["configuration"]

assert config["availability_policy"] == "research_proxy"
assert config["lag_days"] == 120
assert config["lookback_months"] == 3
assert config["min_airline_flights"] == 100

feature_inventory = {
    item["period"]: item
    for item in feature_manifest["files"]
}

source_inventory = {
    item["report_month"]: item
    for item in feature_manifest["source_inventory"]
}

expected_months = set(
    pd.period_range(
        "2023-01",
        "2024-08",
        freq="M",
    ).astype(str)
)

assert len(feature_manifest["files"]) == 20
assert set(feature_inventory) == expected_months

print("=== Feature Manifest ===")

print("Feature version:", feature_manifest["feature_version"])
print("Historical feature count:", len(history_columns))
print("Available target months:", len(feature_inventory))

display(pd.DataFrame([config]))

print("\nFeature manifest validation passed!")

## 2. Model Dataset Construction

Monthly historical feature files are joined with processed BTS records using the established flight candidate key.

Both processed source files and historical feature files are verified against their recorded SHA256 checksums.

The original flight order and target alignment are preserved.

Only scheduled and historical model features are retained in the feature matrix. Flight identifiers, target labels, and audit metadata are excluded from model inputs.

Training and validation datasets are constructed independently according to the predefined temporal split.

In [ ]:
from flight_delay.data.storage import load_parquet

from flight_delay.features.build import (
    build_baseline_features,
)

BASELINE_INPUT_COLUMNS = [
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
]


def load_model_month(month):

    if month not in feature_inventory:
        raise ValueError(
            f"Month outside development scope: {month}"
        )

    year, month_number = month.split("-")

    raw_path = (
        PROCESSED_DIR
        / f"bts_{year}_{month_number}_completed.parquet"
    )

    record = feature_inventory[month]

    history_path = (
        FEATURE_DIR / record["filename"]
    )

    # Ensure that source versions have not changed.
    assert sha256_file(raw_path) == (
        source_inventory[month]["processed_sha256"]
    )

    assert sha256_file(history_path) == record["sha256"]

    raw = load_parquet(raw_path)
    history = pd.read_parquet(history_path)

    assert len(raw) == len(history) == record["row_count"]

    assert history.columns.tolist() == (
        identity_columns + history_columns
    )

    for frame in [raw, history]:
        assert not frame[identity_columns].isna().any().any()
        assert not frame.duplicated(identity_columns).any()

    joined = raw.assign(
        _row_id=np.arange(len(raw))
    ).merge(
        history,
        on=identity_columns,
        how="left",
        validate="one_to_one",
        indicator=True,
        sort=False,
    )

    assert len(joined) == len(raw)
    assert joined["_merge"].eq("both").all()

    joined = (
        joined.sort_values("_row_id")
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        joined[identity_columns],
        raw[identity_columns].reset_index(drop=True),
        check_dtype=False,
    )

    X_base = build_baseline_features(
        joined[BASELINE_INPUT_COLUMNS]
    ).reset_index(drop=True)

    X_history = joined[history_columns].copy()

    missing_rows = int(
        X_history.isna().any(axis=1).sum()
    )

    assert missing_rows == record["missing_feature_rows"]

    for column in history_columns:
        assert X_history[column].dropna().between(0, 1).all()

    X = pd.concat(
        [X_base, X_history],
        axis=1,
    )

    y_dep = joined["DepDel15"].copy()
    y_arr = joined["ArrDel15"].copy()

    assert X.shape == (len(raw), 21)

    np.testing.assert_array_equal(
        y_dep.to_numpy(),
        raw["DepDel15"].to_numpy(),
    )

    np.testing.assert_array_equal(
        y_arr.to_numpy(),
        raw["ArrDel15"].to_numpy(),
    )

    assert y_dep.isin([0, 1]).all()
    assert y_arr.isin([0, 1]).all()

    return {
        "X": X,
        "y_departure": y_dep,
        "y_arrival": y_arr,
        "flight_dates": joined["FlightDate"].copy(),
    }

In [ ]:
def load_model_split(start, end):

    months = pd.period_range(
        start,
        end,
        freq="M",
    ).astype(str)

    parts = []

    for month in months:

        result = load_model_month(month)

        parts.append(result)

        print(
            f"{month} | "
            f"records={len(result['X']):,}"
        )

    return {
        key: pd.concat(
            [part[key] for part in parts],
            ignore_index=True,
        )
        for key in parts[0]
    }


start_time = time.perf_counter()

print("=== Loading Training Dataset ===")

train_v2 = load_model_split(
    "2023-01", "2023-12"
)

print("\n=== Loading Validation Dataset ===")

validation_v2 = load_model_split(
    "2024-01", "2024-08"
)

print("\n=== Dataset Summary ===")

print("Training:", train_v2["X"].shape)
print("Validation:", validation_v2["X"].shape)

print(
    f"Loading time: "
    f"{time.perf_counter() - start_time:.2f} seconds"
)

assert train_v2["X"].shape == (1_478_412, 21)
assert validation_v2["X"].shape == (1_020_082, 21)

assert (
    train_v2["flight_dates"].max()
    < validation_v2["flight_dates"].min()
)

print("\nModel dataset loading passed!")


## 3. Historical Feature Missing Value Treatment

Two training flights have missing historical features because their scheduled departure timestamps are ambiguous during a daylight saving transition.

These records are retained to preserve the experimental cohort.

Median imputation parameters are estimated exclusively from the training dataset and applied consistently to training and validation data.

This is a model preprocessing step rather than a reconstruction of point-in-time historical statistics.

No validation or test information is used to fit the imputation parameters.


In [ ]:
X_train = train_v2["X"]
X_validation = validation_v2["X"]

print("=== Missing Values Before Imputation ===")

missing_before = pd.DataFrame({
    "Training": X_train[history_columns].isna().sum(),
    "Validation": X_validation[history_columns].isna().sum(),
})

display(missing_before)

assert (
    X_train[history_columns].isna().any(axis=1).sum()
    == 2
)

assert (
    X_validation[history_columns].isna().any(axis=1).sum()
    == 0
)

# Fit exclusively on training data.
history_medians = X_train[history_columns].median()

assert history_medians.notna().all()

print("\nTraining-fitted medians:")

display(history_medians)

# Apply the same fitted parameters.
for column in history_columns:

    X_train[column] = X_train[column].fillna(
        history_medians[column]
    )

    X_validation[column] = X_validation[column].fillna(
        history_medians[column]
    )

assert not X_train.isna().any().any()
assert not X_validation.isna().any().any()

assert len(X_train) == 1_478_412
assert len(X_validation) == 1_020_082

print("\n=== Missing Values After Imputation ===")

print("Training:", X_train.isna().sum().sum())
print("Validation:", X_validation.isna().sum().sum())

print("\nTraining-only median imputation passed!")

In [ ]:
baseline_columns = [
    column
    for column in X_train.columns
    if column not in history_columns
]

FEATURE_SETS = {
    "V1": baseline_columns,
    "V2-A": baseline_columns + history_columns[:2],
    "V2-B": baseline_columns + history_columns,
}

assert len(FEATURE_SETS["V1"]) == 17
assert len(FEATURE_SETS["V2-A"]) == 19
assert len(FEATURE_SETS["V2-B"]) == 21

assert X_train.columns.tolist() == X_validation.columns.tolist()

assert len(set(FEATURE_SETS["V2-B"])) == 21

print("=== Ablation Feature Sets ===")

for name, columns in FEATURE_SETS.items():

    print(
        f"\n{name}: {len(columns)} features"
    )

    print(columns)

print("\nAblation feature sets validated!")

In [ ]:
dataset_summary = []

for name, dataset in [
    ("Training", train_v2),
    ("Validation", validation_v2),
]:

    dataset_summary.append({
        "Dataset": name,
        "Records": len(dataset["X"]),
        "DeparturePositiveRate": (
            dataset["y_departure"].mean()
        ),
        "ArrivalPositiveRate": (
            dataset["y_arrival"].mean()
        ),
        "StartDate": dataset["flight_dates"].min(),
        "EndDate": dataset["flight_dates"].max(),
    })

dataset_summary = pd.DataFrame(dataset_summary)

display(dataset_summary)


## 4. Logistic Regression Ablation Study

### Objective

Evaluate the incremental predictive value of historical delay features using Logistic Regression.

### Experimental Configurations

- V1: Scheduled baseline features
- V2-A: Baseline + origin historical features
- V2-B: Baseline + origin and airline-origin historical features

### Controlled Variables

- Identical training and validation periods
- Identical target definitions
- Identical flight cohorts
- Identical classifier hyperparameters
- Identical evaluation metrics
- Training-only preprocessing

### Hypotheses

H1: Origin-level historical delay features improve validation Average Precision relative to the scheduled baseline.

H2: Airline-origin historical features provide additional predictive value beyond origin-level historical features.

### Evaluation

Departure and arrival delay prediction are evaluated independently.

Average Precision is the primary evaluation metric. Threshold-dependent metrics are calculated using a fixed threshold of 0.5.

No hyperparameter tuning or threshold optimization is performed during this experiment.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from flight_delay.models.preprocessing import (
    make_model_preprocessor,
)

base_preprocessor = make_model_preprocessor()

print("=== Existing Baseline Preprocessor ===")
print(base_preprocessor)

In [ ]:
CATEGORICAL_COLUMNS = [
    "Reporting_Airline",
    "Origin",
    "Dest",
]

BASELINE_NUMERIC_COLUMNS = [
    column
    for column in FEATURE_SETS["V1"]
    if column not in CATEGORICAL_COLUMNS
]


def make_ablation_preprocessor(feature_set):

    selected = FEATURE_SETS[feature_set]

    numerical_columns = [
        column
        for column in selected
        if column not in CATEGORICAL_COLUMNS
    ]

    preprocessor = clone(base_preprocessor)

    assert isinstance(
        preprocessor, ColumnTransformer
    )

    updated_transformers = []
    numeric_found = False
    categorical_found = False

    for name, transformer, columns in (
        preprocessor.transformers
    ):

        if set(columns) == set(BASELINE_NUMERIC_COLUMNS):

            updated_transformers.append(
                (name, transformer, numerical_columns)
            )

            numeric_found = True

        elif set(columns) == set(CATEGORICAL_COLUMNS):

            updated_transformers.append(
                (name, transformer, columns)
            )

            categorical_found = True

        else:
            raise ValueError(
                f"Unexpected transformer: {name}"
            )

    assert numeric_found
    assert categorical_found

    preprocessor.set_params(
        transformers=updated_transformers
    )

    return preprocessor


for name in FEATURE_SETS:

    preprocessor = make_ablation_preprocessor(name)

    print(f"\n=== {name} ===")

    for name_, _, columns in preprocessor.transformers:
        print(name_, len(columns), "features")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
)


def evaluate_binary_model(
    y_true,
    probabilities,
    threshold=0.5,
):

    probabilities = np.asarray(
        probabilities, dtype=float
    )

    assert len(y_true) == len(probabilities)
    assert np.isfinite(probabilities).all()

    assert (
        (probabilities >= 0)
        & (probabilities <= 1)
    ).all()

    predictions = (
        probabilities >= threshold
    ).astype(int)

    return {
        "AP": average_precision_score(
            y_true, probabilities
        ),
        "ROC_AUC": roc_auc_score(
            y_true, probabilities
        ),
        "Accuracy": accuracy_score(
            y_true, predictions
        ),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "Brier": brier_score_loss(
            y_true, probabilities
        ),
    }

In [ ]:
import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


LR_PARAMS = {
    "solver": "lbfgs",
    "penalty": "l2",
    "C": 1.0,
    "max_iter": 300,
    "class_weight": None,
}


TARGETS = {
    "Departure": "y_departure",
    "Arrival": "y_arrival",
}


def train_lr_ablation(feature_set, task):

    selected_columns = FEATURE_SETS[feature_set]

    X_tr = X_train[selected_columns]
    X_val = X_validation[selected_columns]

    y_tr = train_v2[TARGETS[task]]
    y_val = validation_v2[TARGETS[task]]

    model = Pipeline([
        (
            "preprocessor",
            make_ablation_preprocessor(feature_set),
        ),
        (
            "classifier",
            LogisticRegression(**LR_PARAMS),
        ),
    ])

    start = time.perf_counter()

    with warnings.catch_warnings(record=True) as caught:

        warnings.simplefilter(
            "always", ConvergenceWarning
        )

        model.fit(X_tr, y_tr)

    training_time = time.perf_counter() - start

    converged = not any(
        issubclass(w.category, ConvergenceWarning)
        for w in caught
    )

    for warning in caught:
        print("Warning:", warning.message)

    assert model.named_steps[
        "classifier"
    ].classes_.tolist() == [0, 1]

    start = time.perf_counter()

    probabilities = model.predict_proba(
        X_val
    )[:, 1]

    inference_time = time.perf_counter() - start

    metrics = evaluate_binary_model(
        y_val,
        probabilities,
    )

    result = {
        "Model": "Logistic Regression",
        "FeatureSet": feature_set,
        "Task": task,
        "FeatureCount": len(selected_columns),
        "TrainingSeconds": training_time,
        "InferenceSeconds": inference_time,
        "Iterations": int(
            model.named_steps[
                "classifier"
            ].n_iter_.max()
        ),
        "Converged": converged,
        **metrics,
    }

    return model, probabilities, result

In [ ]:
lr_models = {}
lr_probabilities = {}
lr_results = []

for task in ["Departure", "Arrival"]:

    print(f"\n=== Training LR V2-A: {task} ===")

    model, probabilities, result = train_lr_ablation(
        feature_set="V2-A",
        task=task,
    )

    lr_models[("V2-A", task)] = model
    lr_probabilities[("V2-A", task)] = probabilities

    lr_results.append(result)

    display(
        pd.DataFrame([result]).round(4)
    )

print("\nLogistic Regression V2-A completed!")

In [ ]:
for task in ["Departure", "Arrival"]:

    print(f"\n=== Training LR V2-B: {task} ===")

    model, probabilities, result = train_lr_ablation(
        feature_set="V2-B",
        task=task,
    )

    lr_models[("V2-B", task)] = model
    lr_probabilities[("V2-B", task)] = probabilities

    lr_results.append(result)

    display(
        pd.DataFrame([result]).round(4)
    )

print("\nLogistic Regression V2-B completed!")

In [ ]:
lr_ablation = pd.DataFrame(lr_results)

metric_columns = [
    "FeatureSet",
    "Task",
    "FeatureCount",
    "AP",
    "ROC_AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Brier",
    "TrainingSeconds",
    "Converged",
]

print("=== Logistic Regression Ablation Results ===")

display(
    lr_ablation[metric_columns].round(4)
)

assert len(lr_ablation) == 4
assert not lr_ablation.duplicated(
    ["FeatureSet", "Task"]
).any()

In [ ]:
# Previously observed V1 validation AP (rounded).
baseline_lr_ap = {
    "Departure": 0.3865,
    "Arrival": 0.3737,
}

lr_ap_comparison = (
    lr_ablation
    .pivot(
        index="Task",
        columns="FeatureSet",
        values="AP",
    )
)

lr_ap_comparison["V1"] = (
    lr_ap_comparison.index.map(baseline_lr_ap)
)

lr_ap_comparison = lr_ap_comparison[
    ["V1", "V2-A", "V2-B"]
]

lr_ap_comparison["OriginGain"] = (
    lr_ap_comparison["V2-A"]
    - lr_ap_comparison["V1"]
)

lr_ap_comparison["AirlineGain"] = (
    lr_ap_comparison["V2-B"]
    - lr_ap_comparison["V2-A"]
)

lr_ap_comparison["TotalGain"] = (
    lr_ap_comparison["V2-B"]
    - lr_ap_comparison["V1"]
)

lr_ap_comparison["RelativeImprovementPct"] = (
    lr_ap_comparison["TotalGain"]
    / lr_ap_comparison["V1"]
    * 100
)

print("=== Incremental Historical Feature Value ===")

display(lr_ap_comparison.round(6))

In [ ]:
import matplotlib.pyplot as plt

ax = lr_ap_comparison[
    ["V1", "V2-A", "V2-B"]
].plot(
    kind="bar",
    figsize=(9, 5),
    rot=0,
)

ax.set_title(
    "Logistic Regression: Historical Feature Ablation"
)

ax.set_ylabel("Validation Average Precision")
ax.set_xlabel("Prediction Task")
ax.set_ylim(0, 1)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.4f",
        padding=3,
    )

plt.legend(title="Feature Set")
plt.tight_layout()
plt.show()

In [ ]:
lr_v1_results = []

for task in ["Departure", "Arrival"]:

    print(f"\n=== Reproducing LR V1: {task} ===")

    model, probabilities, result = train_lr_ablation(
        feature_set="V1",
        task=task,
    )

    lr_models[("V1", task)] = model
    lr_probabilities[("V1", task)] = probabilities
    lr_v1_results.append(result)

    display(pd.DataFrame([result]).round(6))

lr_all_results = pd.concat(
    [
        pd.DataFrame(lr_v1_results),
        lr_ablation,
    ],
    ignore_index=True,
)

print("\nLR baseline reproduction completed!")


## 5. LightGBM Historical Feature Ablation

### Motivation

Logistic Regression experiments showed that adding historical delay features reduced validation Average Precision for both departure and arrival prediction.

LightGBM is evaluated to investigate whether nonlinear relationships and feature interactions can improve the utilization of historical information.

### Experimental Configurations

- V1: Scheduled baseline features
- V2-A: Scheduled and origin historical features
- V2-B: Scheduled, origin and airline-origin historical features

### Controlled Variables

- Identical temporal splits and flight cohorts
- Identical historical feature imputation
- Identical LightGBM hyperparameters
- Training-derived categorical levels
- Identical evaluation metrics

No hyperparameter tuning, probability calibration, or threshold optimization is performed during this ablation experiment.


In [ ]:
from flight_delay.models.lightgbm_baseline import (
    fit_category_levels,
    prepare_lightgbm_features,
)

# Fit categorical levels on training data only.
category_levels = fit_category_levels(
    X_train[FEATURE_SETS["V1"]]
)

lgb_X_train_base = prepare_lightgbm_features(
    X_train[FEATURE_SETS["V1"]],
    category_levels,
)

lgb_X_validation_base = prepare_lightgbm_features(
    X_validation[FEATURE_SETS["V1"]],
    category_levels,
)

assert lgb_X_train_base.index.equals(X_train.index)
assert lgb_X_validation_base.index.equals(X_validation.index)

for column in CATEGORICAL_COLUMNS:

    assert isinstance(
        lgb_X_train_base[column].dtype,
        pd.CategoricalDtype,
    )

    assert (
        lgb_X_train_base[column].dtype
        == lgb_X_validation_base[column].dtype
    )

print("=== LightGBM Feature Preparation ===")
print("Training:", lgb_X_train_base.shape)
print("Validation:", lgb_X_validation_base.shape)

display(lgb_X_train_base.dtypes)

print("\nLightGBM categorical preparation passed!")

In [ ]:
def prepare_lgb_ablation_features(feature_set):

    selected = FEATURE_SETS[feature_set]

    extra_columns = [
        column
        for column in selected
        if column not in FEATURE_SETS["V1"]
    ]

    X_tr = pd.concat(
        [
            lgb_X_train_base,
            X_train[extra_columns],
        ],
        axis=1,
    )[selected]

    X_val = pd.concat(
        [
            lgb_X_validation_base,
            X_validation[extra_columns],
        ],
        axis=1,
    )[selected]

    assert X_tr.columns.tolist() == selected
    assert X_val.columns.tolist() == selected

    assert len(X_tr) == len(X_train)
    assert len(X_val) == len(X_validation)

    assert X_tr.index.equals(X_train.index)
    assert X_val.index.equals(X_validation.index)

    return X_tr, X_val


for feature_set in FEATURE_SETS:

    X_tr, X_val = prepare_lgb_ablation_features(
        feature_set
    )

    print(
        feature_set,
        "| Training:", X_tr.shape,
        "| Validation:", X_val.shape,
    )

    del X_tr, X_val

In [ ]:
from lightgbm import LGBMClassifier

LGB_PARAMS = {
    "objective": "binary",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 100,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": 4,
    "verbosity": -1,
}

print("=== LightGBM Configuration ===")

display(
    pd.Series(LGB_PARAMS, name="Value")
)

In [ ]:
def train_lgb_ablation(feature_set, task):

    X_tr, X_val = prepare_lgb_ablation_features(
        feature_set
    )

    y_tr = train_v2[TARGETS[task]]
    y_val = validation_v2[TARGETS[task]]

    assert X_tr.index.equals(y_tr.index)
    assert X_val.index.equals(y_val.index)

    model = LGBMClassifier(**LGB_PARAMS)

    start = time.perf_counter()

    model.fit(
        X_tr,
        y_tr,
        categorical_feature=CATEGORICAL_COLUMNS,
    )

    training_time = time.perf_counter() - start

    assert model.classes_.tolist() == [0, 1]

    start = time.perf_counter()

    probabilities = model.predict_proba(
        X_val
    )[:, 1]

    inference_time = time.perf_counter() - start

    metrics = evaluate_binary_model(
        y_val,
        probabilities,
    )

    result = {
        "Model": "LightGBM",
        "FeatureSet": feature_set,
        "Task": task,
        "FeatureCount": X_tr.shape[1],
        "TrainingSeconds": training_time,
        "InferenceSeconds": inference_time,
        "Trees": model.n_estimators_,
        **metrics,
    }

    return model, probabilities, result

In [ ]:
lgb_models = {}
lgb_probabilities = {}
lgb_results = []

for feature_set in ["V1", "V2-A", "V2-B"]:

    for task in ["Departure", "Arrival"]:

        print(
            f"\n=== Training LightGBM "
            f"{feature_set}: {task} ==="
        )

        model, probabilities, result = (
            train_lgb_ablation(
                feature_set=feature_set,
                task=task,
            )
        )

        lgb_models[(feature_set, task)] = model

        lgb_probabilities[
            (feature_set, task)
        ] = probabilities

        lgb_results.append(result)

        display(
            pd.DataFrame([result]).round(4)
        )

print("\nLightGBM ablation completed!")

In [ ]:
lgb_ablation = pd.DataFrame(lgb_results)

assert len(lgb_ablation) == 6

assert not lgb_ablation.duplicated(
    ["FeatureSet", "Task"]
).any()

metric_columns = [
    "FeatureSet",
    "Task",
    "FeatureCount",
    "AP",
    "ROC_AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Brier",
    "TrainingSeconds",
]

print("=== LightGBM Ablation Results ===")

display(
    lgb_ablation[metric_columns].round(6)
)

lgb_ap_comparison = lgb_ablation.pivot(
    index="Task",
    columns="FeatureSet",
    values="AP",
)[["V1", "V2-A", "V2-B"]]

lgb_ap_comparison["OriginGain"] = (
    lgb_ap_comparison["V2-A"]
    - lgb_ap_comparison["V1"]
)

lgb_ap_comparison["AirlineGain"] = (
    lgb_ap_comparison["V2-B"]
    - lgb_ap_comparison["V2-A"]
)

lgb_ap_comparison["TotalGain"] = (
    lgb_ap_comparison["V2-B"]
    - lgb_ap_comparison["V1"]
)

lgb_ap_comparison["RelativeImprovementPct"] = (
    lgb_ap_comparison["TotalGain"]
    / lgb_ap_comparison["V1"]
    * 100
)

print("\n=== LightGBM Incremental Feature Value ===")

display(lgb_ap_comparison.round(6))

In [ ]:
all_ablation_results = pd.concat(
    [
        lr_all_results,
        lgb_ablation,
    ],
    ignore_index=True,
)

assert len(all_ablation_results) == 12

assert not all_ablation_results.duplicated(
    ["Model", "FeatureSet", "Task"]
).any()

ap_summary = all_ablation_results.pivot(
    index=["Task", "Model"],
    columns="FeatureSet",
    values="AP",
)[["V1", "V2-A", "V2-B"]]

print("=== Combined Ablation Study ===")

display(ap_summary.round(6))

print("\n=== Validation Model Ranking ===")

display(
    all_ablation_results[
        [
            "Task",
            "Model",
            "FeatureSet",
            "AP",
            "ROC_AUC",
            "F1",
            "Brier",
        ]
    ].sort_values(
        ["Task", "AP"],
        ascending=[True, False],
    ).round(6)
)

In [ ]:
for task in ["Departure", "Arrival"]:

    plot_data = ap_summary.loc[task]

    ax = plot_data.plot(
        kind="bar",
        figsize=(9, 5),
        rot=0,
        width=0.75,
    )

    ax.set_title(
        f"{task} Delay: Historical Feature Ablation"
    )

    ax.set_ylabel("Validation Average Precision")
    ax.set_xlabel("Model")
    ax.set_ylim(0, 0.5)

    for container in ax.containers:
        ax.bar_label(
            container,
            fmt="%.4f",
            padding=3,
        )

    plt.legend(title="Feature Set")
    plt.tight_layout()
    plt.show()


## 6. Temporal Generalization and Feature Diagnostics

### Motivation

Historical delay features reduced overall validation Average Precision for both Logistic Regression and LightGBM.

This section investigates whether the degradation is associated with temporal distribution shifts, delayed historical information, or limitations in feature utilization.

### Diagnostic Methods

- Monthly validation performance comparison
- Monthly historical feature and target mean analysis
- Historical feature-target correlation
- Logistic Regression coefficient inspection
- LightGBM feature importance analysis
- Training versus validation performance comparison

These analyses are exploratory and do not establish causal relationships.

The calibration, threshold-selection, and final test datasets remain untouched.


In [ ]:
validation_months = (
    pd.to_datetime(validation_v2["flight_dates"])
    .dt.to_period("M")
    .astype(str)
)

monthly_results = []

for model_name, probability_store in [
    ("Logistic Regression", lr_probabilities),
    ("LightGBM", lgb_probabilities),
]:

    for feature_set in ["V1", "V2-A", "V2-B"]:

        for task in ["Departure", "Arrival"]:

            probabilities = probability_store[
                (feature_set, task)
            ]

            y_true = validation_v2[
                TARGETS[task]
            ].to_numpy()

            assert len(probabilities) == len(y_true)

            for month in sorted(validation_months.unique()):

                mask = validation_months.eq(month).to_numpy()

                metrics = evaluate_binary_model(
                    y_true[mask],
                    probabilities[mask],
                )

                monthly_results.append({
                    "Month": month,
                    "Model": model_name,
                    "FeatureSet": feature_set,
                    "Task": task,
                    "Records": int(mask.sum()),
                    "PositiveRate": y_true[mask].mean(),
                    **metrics,
                })

monthly_ablation = pd.DataFrame(monthly_results)

assert len(monthly_ablation) == 96

print("=== Monthly Ablation Results ===")

display(monthly_ablation.head(12).round(6))

In [ ]:
monthly_ap_comparison = monthly_ablation.pivot(
    index=["Month", "Model", "Task"],
    columns="FeatureSet",
    values="AP",
).reset_index()

monthly_ap_comparison["OriginGain"] = (
    monthly_ap_comparison["V2-A"]
    - monthly_ap_comparison["V1"]
)

monthly_ap_comparison["AirlineGain"] = (
    monthly_ap_comparison["V2-B"]
    - monthly_ap_comparison["V2-A"]
)

monthly_ap_comparison["TotalGain"] = (
    monthly_ap_comparison["V2-B"]
    - monthly_ap_comparison["V1"]
)

monthly_ap_comparison["V2BImproved"] = (
    monthly_ap_comparison["TotalGain"] > 0
)

print("=== Monthly Historical Feature Gains ===")

display(monthly_ap_comparison.round(6))

monthly_gain_summary = (
    monthly_ap_comparison
    .groupby(["Model", "Task"])
    .agg(
        MonthsImproved=("V2BImproved", "sum"),
        MeanMonthlyGain=("TotalGain", "mean"),
        WorstMonthlyGain=("TotalGain", "min"),
        BestMonthlyGain=("TotalGain", "max"),
    )
)

print("\n=== Monthly Gain Summary ===")

display(monthly_gain_summary.round(6))

In [ ]:
import matplotlib.pyplot as plt

for task in ["Departure", "Arrival"]:

    for model_name in [
        "Logistic Regression",
        "LightGBM",
    ]:

        subset = monthly_ablation.loc[
            monthly_ablation["Task"].eq(task)
            & monthly_ablation["Model"].eq(model_name)
        ]

        plot_data = subset.pivot(
            index="Month",
            columns="FeatureSet",
            values="AP",
        )[["V1", "V2-A", "V2-B"]]

        ax = plot_data.plot(
            figsize=(10, 4),
            marker="o",
        )

        ax.set_title(
            f"{model_name} — {task} Monthly AP"
        )

        ax.set_ylabel("Average Precision")
        ax.set_xlabel("Validation Month")
        ax.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

In [ ]:
drift_results = []

for split_name, dataset in [
    ("Training", train_v2),
    ("Validation", validation_v2),
]:

    months = (
        pd.to_datetime(dataset["flight_dates"])
        .dt.to_period("M")
        .astype(str)
    )

    for month in sorted(months.unique()):

        mask = months.eq(month)

        X_month = dataset["X"].loc[mask]

        for task, origin_feature, airline_feature in [
            (
                "Departure",
                "origin_dep_delay_rate_3m",
                "airline_origin_dep_delay_rate_3m",
            ),
            (
                "Arrival",
                "origin_outbound_arr_delay_rate_3m",
                "airline_origin_arr_delay_rate_3m",
            ),
        ]:

            y_month = dataset[TARGETS[task]].loc[mask]

            actual_rate = y_month.mean()

            origin_history = X_month[
                origin_feature
            ].mean()

            airline_history = X_month[
                airline_feature
            ].mean()

            drift_results.append({
                "Split": split_name,
                "Month": month,
                "Task": task,
                "ActualDelayRate": actual_rate,
                "OriginHistoricalRate": origin_history,
                "AirlineHistoricalRate": airline_history,
                "OriginGap": origin_history - actual_rate,
                "AirlineGap": airline_history - actual_rate,
            })

historical_drift = pd.DataFrame(drift_results)

display(historical_drift.round(6))

In [ ]:
for task in ["Departure", "Arrival"]:

    subset = historical_drift.loc[
        historical_drift["Task"].eq(task)
    ].sort_values("Month")

    ax = subset.set_index("Month")[
        [
            "ActualDelayRate",
            "OriginHistoricalRate",
            "AirlineHistoricalRate",
        ]
    ].plot(
        figsize=(12, 5),
        marker="o",
    )

    ax.axvline(
        x=11.5,
        linestyle="--",
        color="gray",
        label="Train/Validation Boundary",
    )

    ax.set_title(
        f"{task}: Actual vs. Historical Delay Rates"
    )

    ax.set_ylabel("Delay Rate")
    ax.set_xlabel("Month")
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
correlation_results = []

for split_name, dataset in [
    ("Training", train_v2),
    ("Validation", validation_v2),
]:

    X_history = dataset["X"][history_columns]

    for task in ["Departure", "Arrival"]:

        y = dataset[TARGETS[task]]

        for feature in history_columns:

            correlation_results.append({
                "Split": split_name,
                "Task": task,
                "Feature": feature,
                "Correlation": X_history[feature].corr(y),
            })

historical_correlations = pd.DataFrame(
    correlation_results
)

correlation_comparison = (
    historical_correlations.pivot(
        index=["Task", "Feature"],
        columns="Split",
        values="Correlation",
    )
)

correlation_comparison["Change"] = (
    correlation_comparison["Validation"]
    - correlation_comparison["Training"]
)

print("=== Historical Feature Correlations ===")

display(correlation_comparison.round(6))

In [ ]:
coefficient_results = []

for task in ["Departure", "Arrival"]:

    model = lr_models[("V2-B", task)]

    preprocessor = model.named_steps[
        "preprocessor"
    ]

    classifier = model.named_steps[
        "classifier"
    ]

    feature_names = (
        preprocessor.get_feature_names_out()
    )

    coefficients = classifier.coef_.ravel()

    assert len(feature_names) == len(coefficients)

    coefficient_table = pd.DataFrame({
        "EncodedFeature": feature_names,
        "Coefficient": coefficients,
    })

    for feature in history_columns:

        match = coefficient_table.loc[
            coefficient_table["EncodedFeature"].map(
                lambda name: (
                    name == feature
                    or name.endswith("__" + feature)
                )
            )
        ]

        assert len(match) == 1

        coefficient = match["Coefficient"].iloc[0]

        coefficient_results.append({
            "Task": task,
            "Feature": feature,
            "Coefficient": coefficient,
            "OddsRatio": np.exp(coefficient),
        })

lr_history_coefficients = pd.DataFrame(
    coefficient_results
)

print("=== Logistic Regression Historical Coefficients ===")

display(lr_history_coefficients.round(6))

In [ ]:
lgb_importance_results = []

for task in ["Departure", "Arrival"]:

    model = lgb_models[("V2-B", task)]

    booster = model.booster_

    names = booster.feature_name()

    gain = booster.feature_importance(
        importance_type="gain"
    )

    splits = booster.feature_importance(
        importance_type="split"
    )

    importance = pd.DataFrame({
        "Task": task,
        "Feature": names,
        "Gain": gain,
        "SplitCount": splits,
    })

    importance["GainPct"] = (
        importance["Gain"]
        / importance["Gain"].sum()
        * 100
    )

    importance["FeatureGroup"] = np.where(
        importance["Feature"].isin(history_columns),
        "Historical",
        "Scheduled",
    )

    lgb_importance_results.append(importance)

lgb_importance = pd.concat(
    lgb_importance_results,
    ignore_index=True,
)

print("=== Historical Feature Importance ===")

display(
    lgb_importance.loc[
        lgb_importance["FeatureGroup"].eq("Historical")
    ].sort_values(
        ["Task", "GainPct"],
        ascending=[True, False],
    ).round(6)
)

print("\n=== Feature Group Importance ===")

display(
    lgb_importance.groupby(
        ["Task", "FeatureGroup"]
    )["GainPct"].sum().round(4)
)

In [ ]:
for task in ["Departure", "Arrival"]:

    subset = lgb_importance.loc[
        lgb_importance["Task"].eq(task)
    ].sort_values("GainPct")

    ax = subset.plot.barh(
        x="Feature",
        y="GainPct",
        figsize=(10, 8),
        legend=False,
    )

    ax.set_title(
        f"LightGBM V2-B: {task} Feature Importance"
    )

    ax.set_xlabel("Percentage of Total Gain")

    plt.tight_layout()
    plt.show()

In [ ]:
generalization_results = []

for model_name, model_store in [
    ("Logistic Regression", lr_models),
    ("LightGBM", lgb_models),
]:

    for feature_set in ["V1", "V2-A", "V2-B"]:

        selected = FEATURE_SETS[feature_set]

        if model_name == "Logistic Regression":

            X_tr = X_train[selected]

        else:

            X_tr = pd.concat(
                [
                    lgb_X_train_base,
                    X_train[
                        [
                            col for col in selected
                            if col in history_columns
                        ]
                    ],
                ],
                axis=1,
            )[selected]

        for task in ["Departure", "Arrival"]:

            model = model_store[
                (feature_set, task)
            ]

            y_tr = train_v2[TARGETS[task]]

            train_prob = model.predict_proba(
                X_tr
            )[:, 1]

            train_ap = average_precision_score(
                y_tr,
                train_prob,
            )

            val_ap = all_ablation_results.loc[
                all_ablation_results["Model"].eq(model_name)
                & all_ablation_results["FeatureSet"].eq(feature_set)
                & all_ablation_results["Task"].eq(task),
                "AP",
            ].iloc[0]

            generalization_results.append({
                "Model": model_name,
                "FeatureSet": feature_set,
                "Task": task,
                "TrainAP": train_ap,
                "ValidationAP": val_ap,
                "GeneralizationGap": train_ap - val_ap,
            })

            del train_prob

        del X_tr

generalization_summary = pd.DataFrame(
    generalization_results
)

display(generalization_summary.round(6))


## 7. Experimental Conclusions and Model Selection

### Overall Findings

Adding historical delay statistics did not improve
out-of-time validation performance for either Logistic
Regression or LightGBM.

Across both model families and prediction tasks,
the scheduled-feature baseline achieved higher
validation Average Precision than the historical
feature configurations.

### Diagnostic Findings

- Historical delay rates exhibited substantial temporal
  mismatch with actual monthly delay rates.
- Historical feature-target correlations weakened
  between training and validation.
- Historical features accounted for approximately
  10.25% and 11.80% of LightGBM training gain for
  departure and arrival prediction, respectively.
- Historical features improved training AP but reduced
  overall validation AP.
- Some individual validation months benefited from
  historical features, but improvements were inconsistent.

These findings suggest limited temporal stability under
the current historical feature configuration.

They do not establish that historical information is
inherently ineffective or that source availability lag
is the sole cause of performance degradation.

### Selected Models

Departure prediction:
- LightGBM V1
- Validation AP: 0.386799
- Validation ROC-AUC: 0.660694

Arrival prediction:
- Logistic Regression V1
- Validation AP: 0.373736
- Validation ROC-AUC: 0.643443

Both selected models use the original 17 scheduled
features.

The historical feature pipeline is retained as a validated
experimental component but is excluded from the selected
model inputs.

### Limitations

Historical source availability is based on an assumed
120-day research-proxy policy.

The retrospective BTS schedule is not a verified
historical T-24h schedule snapshot.

### Next Steps

Freeze the selected model configurations and proceed
with probability calibration, decision threshold
selection, and an independent 2025 final evaluation.


In [ ]:
ABLATION_DIR = ROOT / "artifacts/historical_ablation"

ABLATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

experiment_tables = {
    "ablation_results": all_ablation_results,
    "monthly_ablation": monthly_ablation,
    "monthly_gain_summary": (
        monthly_gain_summary.reset_index()
    ),
    "historical_drift": historical_drift,
    "historical_correlations": historical_correlations,
    "lr_history_coefficients": lr_history_coefficients,
    "lgb_feature_importance": lgb_importance,
    "generalization_summary": generalization_summary,
}

for name, table in experiment_tables.items():

    path = ABLATION_DIR / f"{name}.csv"

    table.to_csv(path, index=False)

    print(f"Saved: {path.name}")

print("\nHistorical ablation experiment archived!")


## 8. LightGBM Hyperparameter Optimization

### Objective

Improve the temporal generalization performance of LightGBM
through systematic hyperparameter optimization.

### Experimental Setup

- Feature set: V1 scheduled features (17 features)
- Optimization dataset: January–December 2023
- Optimization method: Optuna TPE
- Primary metric: Mean cross-validation Average Precision
- Number of trials: 20 per prediction task
- Random seed: 42

### Temporal Cross-Validation

Three expanding-window folds are constructed within 2023.

Categorical feature levels are fitted independently using
the training subset of each fold.

Departure and arrival models are optimized separately.

### Leakage Control

The 2024 validation dataset is excluded from hyperparameter
search.

Calibration, threshold-selection, and 2025 test datasets
are not used during optimization.

The experiment remains a retrospective evaluation
based on BTS scheduled-data proxies.


In [ ]:
import optuna
import numpy as np
import pandas as pd
import time

from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score

CV_FOLDS = [
    {
        "fold": 1,
        "train_end": "2023-06",
        "val_start": "2023-07",
        "val_end": "2023-08",
    },
    {
        "fold": 2,
        "train_end": "2023-08",
        "val_start": "2023-09",
        "val_end": "2023-10",
    },
    {
        "fold": 3,
        "train_end": "2023-10",
        "val_start": "2023-11",
        "val_end": "2023-12",
    },
]

train_months = (
    pd.to_datetime(train_v2["flight_dates"])
    .dt.to_period("M")
    .astype(str)
)

assert train_months.min() == "2023-01"
assert train_months.max() == "2023-12"

fold_indices = []
fold_summary = []

for fold in CV_FOLDS:

    train_mask = train_months.le(
        fold["train_end"]
    )

    val_mask = train_months.between(
        fold["val_start"],
        fold["val_end"],
    )

    train_idx = np.flatnonzero(
        train_mask.to_numpy()
    )

    val_idx = np.flatnonzero(
        val_mask.to_numpy()
    )

    assert len(train_idx) > 0
    assert len(val_idx) > 0

    assert not np.intersect1d(
        train_idx, val_idx
    ).size

    assert (
        train_months.iloc[train_idx].max()
        < train_months.iloc[val_idx].min()
    )

    fold_indices.append({
        "fold": fold["fold"],
        "train_idx": train_idx,
        "val_idx": val_idx,
    })

    fold_summary.append({
        "Fold": fold["fold"],
        "TrainingEnd": fold["train_end"],
        "ValidationStart": fold["val_start"],
        "ValidationEnd": fold["val_end"],
        "TrainingRecords": len(train_idx),
        "ValidationRecords": len(val_idx),
    })

cv_summary = pd.DataFrame(fold_summary)

display(cv_summary)

print("\nTemporal cross-validation folds passed!")

In [ ]:
from flight_delay.models.lightgbm_baseline import (
    fit_category_levels,
    prepare_lightgbm_features,
)

cv_data = []

for fold in fold_indices:

    tr_idx = fold["train_idx"]
    va_idx = fold["val_idx"]

    X_fold_train = (
        X_train.iloc[tr_idx][FEATURE_SETS["V1"]]
        .reset_index(drop=True)
    )

    X_fold_val = (
        X_train.iloc[va_idx][FEATURE_SETS["V1"]]
        .reset_index(drop=True)
    )

    # Fit category levels on this fold's training data only.
    fold_categories = fit_category_levels(
        X_fold_train
    )

    X_fold_train = prepare_lightgbm_features(
        X_fold_train,
        fold_categories,
    )

    X_fold_val = prepare_lightgbm_features(
        X_fold_val,
        fold_categories,
    )

    cv_data.append({
        "fold": fold["fold"],
        "X_train": X_fold_train,
        "X_val": X_fold_val,
        "Departure": {
            "y_train": train_v2[
                "y_departure"
            ].iloc[tr_idx].to_numpy(),
            "y_val": train_v2[
                "y_departure"
            ].iloc[va_idx].to_numpy(),
        },
        "Arrival": {
            "y_train": train_v2[
                "y_arrival"
            ].iloc[tr_idx].to_numpy(),
            "y_val": train_v2[
                "y_arrival"
            ].iloc[va_idx].to_numpy(),
        },
    })

    print(
        f"Fold {fold['fold']} | "
        f"Training: {X_fold_train.shape} | "
        f"Validation: {X_fold_val.shape}"
    )

print("\nFold-specific preprocessing passed!")

In [ ]:
def evaluate_lgb_cv(params, task):

    fold_scores = []

    for fold in cv_data:

        model = LGBMClassifier(**params)

        model.fit(
            fold["X_train"],
            fold[task]["y_train"],
            categorical_feature=CATEGORICAL_COLUMNS,
        )

        probabilities = model.predict_proba(
            fold["X_val"]
        )[:, 1]

        ap = average_precision_score(
            fold[task]["y_val"],
            probabilities,
        )

        fold_scores.append(float(ap))

    return fold_scores


baseline_cv_results = []

for task in ["Departure", "Arrival"]:

    scores = evaluate_lgb_cv(
        LGB_PARAMS,
        task,
    )

    baseline_cv_results.append({
        "Task": task,
        "Fold1_AP": scores[0],
        "Fold2_AP": scores[1],
        "Fold3_AP": scores[2],
        "MeanAP": np.mean(scores),
        "StdAP": np.std(scores),
    })

baseline_cv = pd.DataFrame(
    baseline_cv_results
)

print("=== Original LightGBM Cross-Validation ===")

display(baseline_cv.round(6))

In [ ]:
def suggest_lgb_params(trial):

    max_depth = trial.suggest_int(
        "max_depth", 3, 10
    )

    num_leaves = trial.suggest_int(
        "num_leaves", 7, 63
    )

    params = {
        **LGB_PARAMS,

        "max_depth": max_depth,

        "num_leaves": min(
            num_leaves, 2 ** max_depth
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            100,
            2000,
            step=100,
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.1,
            log=True,
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            1000,
            step=100,
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0.001,
            10.0,
            log=True,
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.01,
            30.0,
            log=True,
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0,
        ),
    }

    return params

In [ ]:
def make_objective(task):

    def objective(trial):

        params = suggest_lgb_params(trial)

        scores = evaluate_lgb_cv(
            params,
            task,
        )

        mean_ap = float(np.mean(scores))
        std_ap = float(np.std(scores))

        trial.set_user_attr(
            "fold_scores", scores
        )

        trial.set_user_attr(
            "std_ap", std_ap
        )

        trial.set_user_attr(
            "effective_params", params
        )

        return mean_ap

    return objective

In [ ]:
N_TRIALS = 20

optuna_studies = {}

departure_study = optuna.create_study(
    direction="maximize",
    study_name="lightgbm_departure_v1",
    sampler=optuna.samplers.TPESampler(
        seed=42
    ),
)

start = time.perf_counter()

departure_study.optimize(
    make_objective("Departure"),
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

optuna_studies["Departure"] = departure_study

print("\n=== Departure Optimization ===")

print(
    "Best CV AP:",
    departure_study.best_value,
)

print(
    "Optimization seconds:",
    time.perf_counter() - start,
)

display(
    pd.Series(
        departure_study.best_trial.user_attrs[
            "effective_params"
        ]
    )
)

print(
    "Fold scores:",
    departure_study.best_trial.user_attrs[
        "fold_scores"
    ],
)

In [ ]:
arrival_study = optuna.create_study(
    direction="maximize",
    study_name="lightgbm_arrival_v1",
    sampler=optuna.samplers.TPESampler(
        seed=42
    ),
)

start = time.perf_counter()

arrival_study.optimize(
    make_objective("Arrival"),
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

optuna_studies["Arrival"] = arrival_study

print("\n=== Arrival Optimization ===")

print(
    "Best CV AP:",
    arrival_study.best_value,
)

print(
    "Optimization seconds:",
    time.perf_counter() - start,
)

display(
    pd.Series(
        arrival_study.best_trial.user_attrs[
            "effective_params"
        ]
    )
)

print(
    "Fold scores:",
    arrival_study.best_trial.user_attrs[
        "fold_scores"
    ],
)

In [ ]:
optimization_results = []

for task, study in optuna_studies.items():

    baseline_ap = baseline_cv.loc[
        baseline_cv["Task"].eq(task),
        "MeanAP",
    ].iloc[0]

    best_ap = study.best_value

    optimization_results.append({
        "Task": task,
        "BaselineCV_AP": baseline_ap,
        "OptimizedCV_AP": best_ap,
        "AbsoluteGain": best_ap - baseline_ap,
        "RelativeGainPct": (
            (best_ap - baseline_ap)
            / baseline_ap * 100
        ),
        "OptimizedCV_Std": (
            study.best_trial.user_attrs["std_ap"]
        ),
        "BestTrial": study.best_trial.number,
        "CompletedTrials": len([
            trial for trial in study.trials
            if trial.state
            == optuna.trial.TrialState.COMPLETE
        ]),
    })

optimization_summary = pd.DataFrame(
    optimization_results
)

print("=== Hyperparameter Optimization Summary ===")

display(optimization_summary.round(6))

In [ ]:
import matplotlib.pyplot as plt

for task, study in optuna_studies.items():

    trials = study.trials_dataframe()

    trials = trials.loc[
        trials["state"].eq("COMPLETE")
    ].copy()

    baseline_ap = baseline_cv.loc[
        baseline_cv["Task"].eq(task),
        "MeanAP",
    ].iloc[0]

    trials["BestSoFar"] = (
        trials["value"].cummax()
    )

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.scatter(
        trials["number"],
        trials["value"],
        alpha=0.7,
        label="Trial AP",
    )

    ax.plot(
        trials["number"],
        trials["BestSoFar"],
        label="Best AP so far",
    )

    ax.axhline(
        baseline_ap,
        linestyle="--",
        label="Original LightGBM",
    )

    ax.set_title(
        f"{task}: Optuna Optimization History"
    )

    ax.set_xlabel("Trial")
    ax.set_ylabel("Mean Temporal CV AP")

    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


## 9. Optimized LightGBM Out-of-Time Validation

### Objective

Evaluate whether hyperparameters selected through temporal
cross-validation improve performance on the 2024 validation period.

### Experimental Protocol

- Training: January–December 2023
- Validation: January–August 2024
- Features: Original 17 scheduled features
- Hyperparameters: Selected using 2023 temporal CV only
- Primary metric: Average Precision

The optimized models are compared against the original
LightGBM and Logistic Regression baselines.

No further hyperparameter selection is performed using
the 2024 validation results in this experiment.


In [ ]:
OPTUNA_DIR = ROOT / "artifacts/hyperparameter_optimization"

OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

best_params = {}

for task, study in optuna_studies.items():

    best_params[task] = dict(
        study.best_trial.user_attrs["effective_params"]
    )

    study.trials_dataframe().to_csv(
        OPTUNA_DIR / f"{task.lower()}_trials.csv",
        index=False,
    )

with (OPTUNA_DIR / "best_params.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(best_params, f, indent=2)

baseline_cv.to_csv(
    OPTUNA_DIR / "baseline_cv.csv",
    index=False,
)

optimization_summary.to_csv(
    OPTUNA_DIR / "optimization_summary.csv",
    index=False,
)

print("Optuna experiment results saved!")

In [ ]:
optimized_models = {}
optimized_probabilities = {}
optimized_results = []

X_tr = lgb_X_train_base
X_val = lgb_X_validation_base

assert X_tr.shape == (1_478_412, 17)
assert X_val.shape == (1_020_082, 17)

for task in ["Departure", "Arrival"]:

    print(f"\n=== Optimized LightGBM: {task} ===")

    y_tr = train_v2[TARGETS[task]]
    y_val = validation_v2[TARGETS[task]]

    params = best_params[task]

    model = LGBMClassifier(**params)

    assert X_tr.index.equals(y_tr.index)
    assert X_val.index.equals(y_val.index)

    start = time.perf_counter()

    model.fit(
        X_tr,
        y_tr,
        categorical_feature=CATEGORICAL_COLUMNS,
    )

    training_time = time.perf_counter() - start

    assert model.classes_.tolist() == [0, 1]

    start = time.perf_counter()

    val_prob = model.predict_proba(X_val)[:, 1]

    inference_time = time.perf_counter() - start

    train_prob = model.predict_proba(X_tr)[:, 1]

    train_ap = average_precision_score(
        y_tr, train_prob
    )

    metrics = evaluate_binary_model(
        y_val, val_prob
    )

    result = {
        "Model": "LightGBM Tuned",
        "FeatureSet": "V1",
        "Task": task,
        "FeatureCount": X_tr.shape[1],
        "TrainingSeconds": training_time,
        "InferenceSeconds": inference_time,
        "Trees": model.n_estimators_,
        "TrainAP": train_ap,
        "GeneralizationGap": train_ap - metrics["AP"],
        **metrics,
    }

    optimized_models[task] = model
    optimized_probabilities[task] = val_prob
    optimized_results.append(result)

    display(pd.DataFrame([result]).round(6))

    del train_prob

optimized_evaluation = pd.DataFrame(
    optimized_results
)

print("\nOptimized LightGBM evaluation completed!")

In [ ]:
baseline_models = all_ablation_results.loc[
    all_ablation_results["FeatureSet"].eq("V1")
].copy()

baseline_models["Model"] = (
    baseline_models["Model"] + " Original"
)

baseline_vs_tuned = pd.concat(
    [
        baseline_models,
        optimized_evaluation,
    ],
    ignore_index=True,
)

comparison_columns = [
    "Task",
    "Model",
    "AP",
    "ROC_AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Brier",
    "TrainingSeconds",
]

print("=== Original vs. Optimized Models ===")

display(
    baseline_vs_tuned[comparison_columns]
    .sort_values(
        ["Task", "AP"],
        ascending=[True, False],
    )
    .round(6)
)

In [ ]:
tuning_comparison = []

for task in ["Departure", "Arrival"]:

    original = baseline_vs_tuned.loc[
        baseline_vs_tuned["Task"].eq(task)
        & baseline_vs_tuned["Model"].eq(
            "LightGBM Original"
        )
    ].iloc[0]

    tuned = optimized_evaluation.loc[
        optimized_evaluation["Task"].eq(task)
    ].iloc[0]

    previous_best = baseline_models.loc[
        baseline_models["Task"].eq(task),
        "AP",
    ].max()

    tuning_comparison.append({
        "Task": task,
        "OriginalLGB_AP": original["AP"],
        "TunedLGB_AP": tuned["AP"],
        "AbsoluteGain": tuned["AP"] - original["AP"],
        "RelativeGainPct": (
            (tuned["AP"] / original["AP"] - 1) * 100
        ),
        "PreviousBestAP": previous_best,
        "GainOverPreviousBest": (
            tuned["AP"] - previous_best
        ),
        "ROC_AUC_Gain": (
            tuned["ROC_AUC"] - original["ROC_AUC"]
        ),
        "BrierImprovement": (
            original["Brier"] - tuned["Brier"]
        ),
    })

tuning_comparison = pd.DataFrame(
    tuning_comparison
)

print("=== Out-of-Time Optimization Gains ===")

display(tuning_comparison.round(6))

In [ ]:
original_generalization = (
    generalization_summary.loc[
        generalization_summary["Model"].eq("LightGBM")
        & generalization_summary["FeatureSet"].eq("V1")
    ].copy()
)

original_generalization["Model"] = "LightGBM Original"

tuned_generalization = optimized_evaluation[
    [
        "Model",
        "Task",
        "TrainAP",
        "AP",
        "GeneralizationGap",
    ]
].rename(
    columns={"AP": "ValidationAP"}
)

tuning_generalization = pd.concat(
    [
        original_generalization,
        tuned_generalization,
    ],
    ignore_index=True,
)

display(
    tuning_generalization[
        [
            "Model",
            "Task",
            "TrainAP",
            "ValidationAP",
            "GeneralizationGap",
        ]
    ].round(6)
)

In [ ]:
tuned_monthly_results = []

for task in ["Departure", "Arrival"]:

    probabilities = optimized_probabilities[task]

    y_true = validation_v2[
        TARGETS[task]
    ].to_numpy()

    for month in sorted(validation_months.unique()):

        mask = validation_months.eq(month).to_numpy()

        tuned_monthly_results.append({
            "Month": month,
            "Task": task,
            "Model": "LightGBM Tuned",
            **evaluate_binary_model(
                y_true[mask],
                probabilities[mask],
            ),
        })

tuned_monthly = pd.DataFrame(
    tuned_monthly_results
)

original_monthly = monthly_ablation.loc[
    monthly_ablation["FeatureSet"].eq("V1")
].copy()

original_monthly["Model"] += " Original"

monthly_tuning_comparison = pd.concat(
    [
        original_monthly,
        tuned_monthly,
    ],
    ignore_index=True,
)

for task in ["Departure", "Arrival"]:

    plot_data = monthly_tuning_comparison.loc[
        monthly_tuning_comparison["Task"].eq(task)
    ].pivot(
        index="Month",
        columns="Model",
        values="AP",
    )

    ax = plot_data.plot(
        figsize=(10, 4),
        marker="o",
    )

    ax.set_title(
        f"{task}: Monthly AP After Hyperparameter Tuning"
    )

    ax.set_ylabel("Average Precision")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
optimized_evaluation.to_csv(
    OPTUNA_DIR / "optimized_evaluation.csv",
    index=False,
)

tuning_comparison.to_csv(
    OPTUNA_DIR / "tuning_comparison.csv",
    index=False,
)

tuned_monthly.to_csv(
    OPTUNA_DIR / "tuned_monthly.csv",
    index=False,
)


## 10. Advanced Scheduled Feature Engineering

### Motivation

Hyperparameter optimization improved out-of-time
validation performance and reduced model complexity.

However, the predictive information available in the
original scheduled feature set remains limited.

### Experimental Configurations

- V1 Tuned: 17 scheduled features
- V3-A: V1 + route interactions and clock-hour features
- V3-B: V3-A + federal holiday proximity features

### New Features

- Route
- Airline-route interaction
- Scheduled departure hour
- Scheduled arrival hour
- US federal holiday indicator
- Days since the previous federal holiday
- Days until the next federal holiday

### Leakage Control

All features are derived from scheduled flight information
or deterministic calendar information.

No target labels or actual flight outcomes are used
during feature construction.

Categorical levels will be fitted exclusively on
the corresponding training partition.

The retrospective schedule-proxy limitation remains.


In [ ]:
from flight_delay.features.advanced_scheduled import (
    ADVANCED_FEATURE_COLUMNS,
    INTERACTION_FEATURE_COLUMNS,
    HOLIDAY_FEATURE_COLUMNS,
    build_advanced_scheduled_features,
)

advanced_examples = pd.DataFrame({
    "FlightDate": [
        "2023-07-03",
        "2023-07-04",
        "2023-07-05",
        "2023-12-31",
    ],
    "Origin": ["SFO", "SFO", "ATL", "DEN"],
    "Dest": ["JFK", "JFK", "LAX", "ORD"],
    "Reporting_Airline": ["UA", "DL", "DL", "UA"],
    "CRSDepTime": [1730, 800, 30, 2400],
    "CRSArrTime": [155, 1650, 230, 0],
})

example_features = build_advanced_scheduled_features(
    advanced_examples
)

display(
    pd.concat(
        [advanced_examples, example_features],
        axis=1,
    )
)

assert example_features.loc[0, "Route"] == "SFO_JFK"

assert example_features.loc[
    0, "Airline_Route"
] == "UA_SFO_JFK"

assert example_features.loc[
    0, "Departure_Hour"
] == 17

assert example_features.loc[
    0, "Arrival_Hour"
] == 1

assert example_features.loc[
    3, "Departure_Hour"
] == 0

assert example_features[
    "is_us_federal_holiday"
].tolist() == [0, 1, 0, 0]

assert example_features.loc[
    0, "days_until_us_federal_holiday"
] == 1

assert example_features.loc[
    2, "days_since_us_federal_holiday"
] == 1

assert example_features.loc[
    3, "days_until_us_federal_holiday"
] == 1

print("\nAdvanced scheduled feature checks passed!")

In [ ]:
jan_raw = load_parquet(
    PROCESSED_DIR / "bts_2023_01_completed.parquet"
)

jan_advanced = build_advanced_scheduled_features(
    jan_raw
)

assert jan_advanced.shape == (112_548, 7)

assert jan_advanced.index.equals(
    jan_raw.index
)

assert not jan_advanced.isna().any().any()

print("=== January Advanced Scheduled Features ===")

print("Feature matrix:", jan_advanced.shape)

print(
    "Unique routes:",
    jan_advanced["Route"].nunique(),
)

print(
    "Unique airline-route combinations:",
    jan_advanced["Airline_Route"].nunique(),
)

print(
    "Federal holiday flights:",
    jan_advanced["is_us_federal_holiday"].sum(),
)

display(jan_advanced.head(10))

display(jan_advanced.describe(include="all"))

print("\nJanuary advanced feature pilot passed!")

In [ ]:
def generate_advanced_split(dataset):

    months = (
        pd.to_datetime(dataset["flight_dates"])
        .dt.to_period("M")
        .astype(str)
    )

    parts = []

    for month in sorted(months.unique()):

        positions = np.flatnonzero(
            months.eq(month).to_numpy()
        )

        year, month_number = month.split("-")

        path = (
            PROCESSED_DIR
            / f"bts_{year}_{month_number}_completed.parquet"
        )

        # Verify the processed source version.
        assert sha256_file(path) == (
            source_inventory[month]["processed_sha256"]
        )

        raw = load_parquet(path)

        assert len(raw) == len(positions)

        # Confirm baseline feature alignment.
        rebuilt_baseline = build_baseline_features(
            raw[BASELINE_INPUT_COLUMNS]
        ).reset_index(drop=True)

        expected_baseline = (
            dataset["X"]
            .iloc[positions][FEATURE_SETS["V1"]]
            .reset_index(drop=True)
        )

        pd.testing.assert_frame_equal(
            rebuilt_baseline,
            expected_baseline,
            check_dtype=False,
        )

        # Confirm date and target alignment.
        np.testing.assert_array_equal(
            raw["FlightDate"].to_numpy(),
            dataset["flight_dates"]
            .iloc[positions].to_numpy(),
        )

        for raw_target, dataset_target in [
            ("DepDel15", "y_departure"),
            ("ArrDel15", "y_arrival"),
        ]:
            np.testing.assert_array_equal(
                raw[raw_target].to_numpy(),
                dataset[dataset_target]
                .iloc[positions].to_numpy(),
            )

        advanced = build_advanced_scheduled_features(
            raw
        )

        advanced.index = dataset["X"].index[positions]

        assert not advanced.isna().any().any()

        parts.append(advanced)

        print(
            f"{month} | "
            f"records={len(advanced):,} | "
            f"features={advanced.shape[1]}"
        )

        del raw, rebuilt_baseline, expected_baseline

    result = pd.concat(parts).sort_index()

    assert result.index.equals(dataset["X"].index)

    assert len(result) == len(dataset["X"])

    return result

In [ ]:
start = time.perf_counter()

print("=== Advanced Training Features ===")

advanced_train = generate_advanced_split(
    train_v2
)

print("\n=== Advanced Validation Features ===")

advanced_validation = generate_advanced_split(
    validation_v2
)

assert advanced_train.shape == (1_478_412, 7)

assert advanced_validation.shape == (1_020_082, 7)

print("\n=== Advanced Feature Summary ===")

print("Training:", advanced_train.shape)
print("Validation:", advanced_validation.shape)

print(
    "Generation seconds:",
    round(time.perf_counter() - start, 2),
)

print("\nFull advanced feature generation passed!")

In [ ]:
coverage_results = []

for column in ["Route", "Airline_Route"]:

    known_categories = set(
        advanced_train[column].unique()
    )

    unknown = ~advanced_validation[column].isin(
        known_categories
    )

    coverage_results.append({
        "Feature": column,
        "TrainingCategories": (
            advanced_train[column].nunique()
        ),
        "ValidationCategories": (
            advanced_validation[column].nunique()
        ),
        "UnknownValidationCategories": (
            advanced_validation.loc[
                unknown, column
            ].nunique()
        ),
        "UnknownValidationRecords": int(
            unknown.sum()
        ),
        "UnknownRatePct": (
            unknown.mean() * 100
        ),
    })

advanced_category_coverage = pd.DataFrame(
    coverage_results
)

print("=== Advanced Categorical Coverage ===")

display(
    advanced_category_coverage.round(6)
)

In [ ]:
V3_FEATURE_SETS = {
    "V1": FEATURE_SETS["V1"],

    "V3-A": (
        FEATURE_SETS["V1"]
        + list(INTERACTION_FEATURE_COLUMNS)
    ),

    "V3-B": (
        FEATURE_SETS["V1"]
        + list(ADVANCED_FEATURE_COLUMNS)
    ),
}

X_train_v3 = pd.concat(
    [
        X_train[FEATURE_SETS["V1"]],
        advanced_train,
    ],
    axis=1,
)

X_validation_v3 = pd.concat(
    [
        X_validation[FEATURE_SETS["V1"]],
        advanced_validation,
    ],
    axis=1,
)

assert X_train_v3.shape == (1_478_412, 24)

assert X_validation_v3.shape == (1_020_082, 24)

assert not X_train_v3.isna().any().any()

assert not X_validation_v3.isna().any().any()

assert (
    X_train_v3.columns.tolist()
    == X_validation_v3.columns.tolist()
)

assert len(set(X_train_v3.columns)) == 24

V3_CATEGORICAL_COLUMNS = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Route",
    "Airline_Route",
]

for name, columns in V3_FEATURE_SETS.items():

    print(
        f"{name} | "
        f"Training: {X_train_v3[columns].shape} | "
        f"Validation: {X_validation_v3[columns].shape}"
    )

print("\nAdvanced feature sets validated!")


## 11. Advanced Scheduled Feature Ablation

### Objective

Evaluate whether explicit route interactions, scheduled
clock-hour features, and holiday proximity features
improve flight delay prediction.

### Experimental Configurations

- V1: Original scheduled features
- V3-A: V1 + route interactions and clock-hour features
- V3-B: V3-A + federal holiday features

### Controlled Variables

- Identical flight cohorts and target definitions
- Previously optimized LightGBM hyperparameters
- Identical temporal cross-validation folds
- Training-only categorical encoding
- Average Precision as the primary metric

### Evaluation Protocol

First, compare feature configurations using the three
previously defined temporal CV folds within 2023.

Then evaluate the fixed configurations using the
2024 January–August development validation dataset.

No additional hyperparameter optimization is conducted.

The calibration, threshold-selection, and final test
datasets remain excluded.


In [ ]:
def prepare_advanced_lgb_features(
    train_frame,
    validation_frame,
    feature_set,
):

    selected = V3_FEATURE_SETS[feature_set]

    base_columns = V3_FEATURE_SETS["V1"]

    # Preserve the original baseline encoding.
    levels = fit_category_levels(
        train_frame[base_columns]
    )

    X_tr = prepare_lightgbm_features(
        train_frame[base_columns],
        levels,
    )

    X_val = prepare_lightgbm_features(
        validation_frame[base_columns],
        levels,
    )

    extra_columns = [
        column for column in selected
        if column not in base_columns
    ]

    X_tr = pd.concat(
        [X_tr, train_frame[extra_columns]],
        axis=1,
    )[selected]

    X_val = pd.concat(
        [X_val, validation_frame[extra_columns]],
        axis=1,
    )[selected]

    categorical_columns = [
        column for column in V3_CATEGORICAL_COLUMNS
        if column in selected
    ]

    # Fit new categorical levels on training data only.
    for column in ["Route", "Airline_Route"]:

        if column not in selected:
            continue

        categories = sorted(
            X_tr[column].unique().tolist()
        )

        dtype = pd.CategoricalDtype(
            categories=categories
        )

        for frame in [X_tr, X_val]:

            values = frame[column].astype("string")

            values = values.where(
                values.isin(categories),
                pd.NA,
            )

            frame[column] = values.astype(dtype)

    assert X_tr.columns.tolist() == selected
    assert X_val.columns.tolist() == selected

    assert X_tr.index.equals(train_frame.index)
    assert X_val.index.equals(validation_frame.index)

    for column in categorical_columns:

        assert isinstance(
            X_tr[column].dtype,
            pd.CategoricalDtype,
        )

        assert X_tr[column].dtype == X_val[column].dtype

        assert X_tr[column].notna().all()

    numeric_columns = [
        column for column in selected
        if column not in categorical_columns
    ]

    for frame in [X_tr, X_val]:
        assert np.isfinite(
            frame[numeric_columns].to_numpy(dtype=float)
        ).all()

    return X_tr, X_val, categorical_columns

In [ ]:
X_tr_check, X_val_check, categorical_check = (
    prepare_advanced_lgb_features(
        X_train_v3,
        X_validation_v3,
        "V3-B",
    )
)

print("=== Advanced LightGBM Encoding ===")

print("Categorical features:", categorical_check)

display(
    X_tr_check.dtypes
)

print("\nUnknown validation categories:")

for column in categorical_check:

    print(
        column,
        int(X_val_check[column].isna().sum()),
    )

assert X_val_check["Route"].isna().sum() == 1950

assert X_val_check["Airline_Route"].isna().sum() == 11863

print("\nAdvanced categorical encoding passed!")

del X_tr_check, X_val_check

In [ ]:
advanced_cv_results = []

# Reuse the CV results of the selected V1 parameters.
for task, study in optuna_studies.items():

    scores = study.best_trial.user_attrs["fold_scores"]

    assert len(scores) == 3

    for fold_number, score in enumerate(scores, start=1):

        advanced_cv_results.append({
            "FeatureSet": "V1",
            "Task": task,
            "Fold": fold_number,
            "AP": score,
        })


# Process one temporal fold at a time.
for fold in fold_indices:

    tr_idx = fold["train_idx"]
    va_idx = fold["val_idx"]

    for feature_set in ["V3-A", "V3-B"]:

        X_tr, X_val, cat_columns = (
            prepare_advanced_lgb_features(
                X_train_v3.iloc[tr_idx],
                X_train_v3.iloc[va_idx],
                feature_set,
            )
        )

        for task in ["Departure", "Arrival"]:

            y = train_v2[TARGETS[task]]

            y_tr = y.iloc[tr_idx]
            y_val = y.iloc[va_idx]

            assert X_tr.index.equals(y_tr.index)
            assert X_val.index.equals(y_val.index)

            model = LGBMClassifier(
                **best_params[task]
            )

            model.fit(
                X_tr,
                y_tr,
                categorical_feature=cat_columns,
            )

            probabilities = model.predict_proba(
                X_val
            )[:, 1]

            ap = average_precision_score(
                y_val,
                probabilities,
            )

            advanced_cv_results.append({
                "FeatureSet": feature_set,
                "Task": task,
                "Fold": fold["fold"],
                "AP": ap,
            })

            print(
                f"Fold {fold['fold']} | "
                f"{feature_set} | "
                f"{task} | AP={ap:.6f}"
            )

            del model, probabilities

        del X_tr, X_val

advanced_cv = pd.DataFrame(advanced_cv_results)

assert len(advanced_cv) == 18

assert not advanced_cv.duplicated(
    ["FeatureSet", "Task", "Fold"]
).any()

print("\nAdvanced temporal CV completed!")

In [ ]:
advanced_cv_summary = (
    advanced_cv
    .groupby(["Task", "FeatureSet"])
    .agg(
        MeanAP=("AP", "mean"),
        StdAP=("AP", "std"),
        MinAP=("AP", "min"),
        MaxAP=("AP", "max"),
    )
    .reset_index()
)

display(advanced_cv_summary.round(6))

advanced_cv_comparison = (
    advanced_cv_summary.pivot(
        index="Task",
        columns="FeatureSet",
        values="MeanAP",
    )[["V1", "V3-A", "V3-B"]]
)

advanced_cv_comparison["InteractionGain"] = (
    advanced_cv_comparison["V3-A"]
    - advanced_cv_comparison["V1"]
)

advanced_cv_comparison["HolidayGain"] = (
    advanced_cv_comparison["V3-B"]
    - advanced_cv_comparison["V3-A"]
)

advanced_cv_comparison["TotalGain"] = (
    advanced_cv_comparison["V3-B"]
    - advanced_cv_comparison["V1"]
)

print("=== Advanced Feature CV Comparison ===")

display(advanced_cv_comparison.round(6))

In [ ]:
advanced_models = {}
advanced_probabilities = {}
advanced_results = []

for feature_set in ["V3-A", "V3-B"]:

    X_tr, X_val, cat_columns = (
        prepare_advanced_lgb_features(
            X_train_v3,
            X_validation_v3,
            feature_set,
        )
    )

    for task in ["Departure", "Arrival"]:

        print(
            f"\n=== Training {feature_set}: {task} ==="
        )

        y_tr = train_v2[TARGETS[task]]
        y_val = validation_v2[TARGETS[task]]

        assert X_tr.index.equals(y_tr.index)
        assert X_val.index.equals(y_val.index)

        model = LGBMClassifier(
            **best_params[task]
        )

        start = time.perf_counter()

        model.fit(
            X_tr,
            y_tr,
            categorical_feature=cat_columns,
        )

        train_seconds = time.perf_counter() - start

        assert model.classes_.tolist() == [0, 1]

        start = time.perf_counter()

        val_prob = model.predict_proba(X_val)[:, 1]

        inference_seconds = time.perf_counter() - start

        train_prob = model.predict_proba(X_tr)[:, 1]

        train_ap = average_precision_score(
            y_tr, train_prob
        )

        metrics = evaluate_binary_model(
            y_val, val_prob
        )

        result = {
            "Model": "LightGBM Tuned",
            "FeatureSet": feature_set,
            "Task": task,
            "FeatureCount": X_tr.shape[1],
            "TrainingSeconds": train_seconds,
            "InferenceSeconds": inference_seconds,
            "TrainAP": train_ap,
            "GeneralizationGap": (
                train_ap - metrics["AP"]
            ),
            **metrics,
        }

        advanced_models[(feature_set, task)] = model

        advanced_probabilities[
            (feature_set, task)
        ] = val_prob

        advanced_results.append(result)

        display(pd.DataFrame([result]).round(6))

        del train_prob

    del X_tr, X_val

advanced_evaluation = pd.DataFrame(advanced_results)

assert len(advanced_evaluation) == 4

print("\nAdvanced model training completed!")

In [ ]:
advanced_comparison = pd.concat(
    [
        optimized_evaluation,
        advanced_evaluation,
    ],
    ignore_index=True,
)

comparison_columns = [
    "Task",
    "FeatureSet",
    "FeatureCount",
    "AP",
    "ROC_AUC",
    "Precision",
    "Recall",
    "F1",
    "Brier",
    "TrainAP",
    "GeneralizationGap",
]

display(
    advanced_comparison[comparison_columns]
    .sort_values(
        ["Task", "AP"],
        ascending=[True, False],
    )
    .round(6)
)

advanced_ap_comparison = (
    advanced_comparison.pivot(
        index="Task",
        columns="FeatureSet",
        values="AP",
    )[["V1", "V3-A", "V3-B"]]
)

advanced_ap_comparison["InteractionGain"] = (
    advanced_ap_comparison["V3-A"]
    - advanced_ap_comparison["V1"]
)

advanced_ap_comparison["HolidayGain"] = (
    advanced_ap_comparison["V3-B"]
    - advanced_ap_comparison["V3-A"]
)

advanced_ap_comparison["TotalGain"] = (
    advanced_ap_comparison["V3-B"]
    - advanced_ap_comparison["V1"]
)

advanced_ap_comparison["RelativeGainPct"] = (
    advanced_ap_comparison["TotalGain"]
    / advanced_ap_comparison["V1"]
    * 100
)

print("=== Advanced Feature Incremental Value ===")

display(advanced_ap_comparison.round(6))

In [ ]:
import matplotlib.pyplot as plt

ax = advanced_ap_comparison[
    ["V1", "V3-A", "V3-B"]
].plot(
    kind="bar",
    figsize=(9, 5),
    rot=0,
)

ax.set_title(
    "Tuned LightGBM: Advanced Feature Ablation"
)

ax.set_ylabel("Validation Average Precision")
ax.set_xlabel("Prediction Task")

ax.set_ylim(
    0,
    max(
        0.5,
        advanced_ap_comparison[
            ["V1", "V3-A", "V3-B"]
        ].max().max() * 1.15,
    ),
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.4f",
        padding=3,
    )

plt.legend(title="Feature Set")
plt.tight_layout()
plt.show()

In [ ]:
ADVANCED_DIR = ROOT / "artifacts/advanced_features"

ADVANCED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for name, table in {
    "category_coverage": advanced_category_coverage,
    "cv_results": advanced_cv,
    "cv_summary": advanced_cv_summary,
    "cv_comparison": advanced_cv_comparison.reset_index(),
    "validation_results": advanced_comparison,
    "ap_comparison": advanced_ap_comparison.reset_index(),
}.items():

    table.to_csv(
        ADVANCED_DIR / f"{name}.csv",
        index=False,
    )

print("Advanced feature experiment results saved!")